AI Crown Design — conditional 3D U-Net occupancy completion
=============================================================
Thesis: "Assessment of Accuracy of AI versus conventional Digital Design of
fixed dental restoration." Reference / gold standard = exocad `design.stl`.

Pipeline (run cells top to bottom on a Kaggle GPU notebook, internet ON):
  1. config            2. constructionInfo parser   3. preprocess -> voxel cache
  4. dataset/augment   5. 3D U-Net                   6. train
  7. inference + mesh  8. evaluate (IoU / Hausdorff / RMS) -> metrics.csv

Input layout expected at INPUT_ROOT, one folder per case (as inspected):
  <case>/upper.stl  lower.stl  design.stl  constructionInfo  [tooth_model.obj ...]
The prepared arch is chosen from the target tooth's FDI quadrant
(1-2 -> upper, 3-4 -> lower); the other arch is the antagonist. Both arches are
already articulated in one world frame, so the antagonist gives the real
opposing profile the protocol asks the model to learn from.

NOTE: this file was authored without a GPU/data sandbox; run cell-by-cell on
Kaggle and expect to tune RES / channels / epochs. Nothing here needs the
local `dcprep` package — it is self-contained.


In [5]:
# %% setup ---------------------------------------------------------------------
!pip -q install trimesh scikit-image rtree manifold3d
import os, glob, json, math, random, re, xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np

SEED = 1337
random.seed(SEED); np.random.seed(SEED)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 12.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.7 MB/s eta 0:00:00


In [6]:
# %% 1. config -----------------------------------------------------------------
class CFG:
    INPUT_ROOT = "/kaggle/input/single-crown"      # mounts here when the dataset is attached
    CACHE      = "/kaggle/working/cache"           # voxel .npz cache
    CKPT       = "/kaggle/working/ckpt"
    OUT        = "/kaggle/working/out"

    ROI_MM     = 20.0        # cube side around the tooth centre (covers crown+context)
    RES        = 128         # final thesis resolution (0.156 mm / voxel)
    PITCH      = ROI_MM / RES

    # input channels: prep-arch shell, antagonist shell  (add neighbours later if wanted)
    IN_CH      = 3           # prep shell + antagonist shell + margin channel
    SPLIT      = (0.60, 0.10, 0.30)   # train / val / test  (by ORIGINAL case, no leakage)

    EPOCHS     = 100
    BATCH      = 1
    LR         = 5e-4
    BASE_CH    = 16          # U-Net width
    AMP        = True
    TOL_MM     = 0.10        # clinical equivalence margin for the stats write-up
    THR        = 0.5         # inference threshold

for d in (CFG.CACHE, CFG.CKPT, CFG.OUT):
    os.makedirs(d, exist_ok=True)


In [7]:
# %% 2. constructionInfo parser ------------------------------------------------
# Minimal port of the validated dcprep parser. Pulls, per tooth: FDI number,
# Center, and the 4x4 ToothModelMatrix (row-vector convention: world = [x y z 1] @ M),
# whose inverse canonicalises the tooth into a centred, axis-aligned frame.

def _f(node, tag, default=None):
    e = node.find(tag)
    return float(e.text) if e is not None and e.text not in (None, "") else default

def _matrix(node, tag):
    m = node.find(tag)
    if m is None:
        return None
    M = np.eye(4)
    for r in range(4):
        for c in range(4):
            e = m.find(f"_{r}{c}")
            if e is not None and e.text is not None:
                M[r, c] = float(e.text)
    return M

def parse_construction_info(path):
    root = ET.parse(path).getroot()          # root tag is the misspelled <ConstuctionInfo>
    teeth = []
    for t in root.findall("./Teeth/Tooth"):
        num = t.find("Number")
        if num is None:
            continue
        fdi = int(num.text)
        ctr = t.find("Center")
        center = np.array([_f(ctr, "x", 0), _f(ctr, "y", 0), _f(ctr, "z", 0)]) if ctr is not None else None
        margin = np.array([[_f(v, "x", 0), _f(v, "y", 0), _f(v, "z", 0)]
                           for v in t.findall("./Margin/Vec3")], float)   # world coords
        teeth.append(dict(fdi=fdi, center=center,
                          tmm=_matrix(t, "ToothModelMatrix"),
                          margin=margin if len(margin) else None))
    return teeth

def world_to_canonical(verts, tmm):
    """world (N,3) -> canonical frame via inverse ToothModelMatrix (row-vector convention)."""
    Minv = np.linalg.inv(tmm)
    h = np.c_[verts, np.ones(len(verts))]
    return (h @ Minv)[:, :3]

def fdi_arch(fdi):
    """upper if quadrant 1/2, lower if 3/4 (and deciduous 5/6 upper, 7/8 lower)."""
    q = fdi // 10
    return "upper" if q in (1, 2, 5, 6) else "lower"


In [8]:
# %% 2b. margin channel -------------------------------------------------------
from scipy.ndimage import binary_dilation

def sample_margin_polyline(M, step=None):
    """Densely resample the 3D margin polyline (canonical coords) so the channel
    is continuous rather than sparse points."""
    if M is None or len(M) < 3:
        return None
    M = np.asarray(M, float)
    step = step or CFG.PITCH / 2
    pts = []
    n = len(M)
    for i in range(n):
        p0, p1 = M[i], M[(i + 1) % n]
        k = max(2, int(np.ceil(np.linalg.norm(p1 - p0) / step)))
        for j in range(k):
            t = j / k
            pts.append((1 - t) * p0 + t * p1)
    return np.asarray(pts, float)

def margin_channel_from_points(margin_canon, radius_vox=2):
    """Rasterise the margin polyline into a thin binary voxel ring."""
    out = np.zeros((CFG.RES,) * 3, np.float32)
    pts = sample_margin_polyline(margin_canon)
    if pts is None or len(pts) == 0:
        return out
    idx = np.round(pts / CFG.PITCH + CFG.RES // 2).astype(int)
    ok = np.all((idx >= 0) & (idx < CFG.RES), axis=1)
    idx = idx[ok]
    if len(idx) == 0:
        return out
    out[idx[:, 0], idx[:, 1], idx[:, 2]] = 1.0
    if radius_vox > 0:
        out = binary_dilation(out, iterations=radius_vox).astype(np.float32)
    return out


In [9]:
# %% 3. preprocess -> voxel cache ---------------------------------------------
import trimesh
from trimesh.voxel import creation as vcreate

def _load(path):
    m = trimesh.load(path, process=False)
    if isinstance(m, trimesh.Scene):
        m = trimesh.util.concatenate(tuple(m.geometry.values()))
    return m

def _voxelise(verts_faces, fill):
    """Rasterise a mesh (already in canonical mm coords) onto the fixed ROI grid.
    Returns a (RES,RES,RES) float array. `fill` -> solid interior (for the crown)."""
    v, f = verts_faces
    mesh = trimesh.Trimesh(vertices=v, faces=f, process=False)
    r = CFG.RES // 2
    vg = vcreate.local_voxelize(mesh, point=np.zeros(3), pitch=CFG.PITCH,
                                radius=r, fill=fill)
    if vg is None:
        return np.zeros((CFG.RES,) * 3, np.float32)
    grid = vg.matrix.astype(np.float32)            # (2r+1)^3 centred on origin
    # crop/pad to exactly RES^3
    out = np.zeros((CFG.RES,) * 3, np.float32)
    s = [min(CFG.RES, grid.shape[i]) for i in range(3)]
    out[:s[0], :s[1], :s[2]] = grid[:s[0], :s[1], :s[2]]
    return out

def preprocess():
    # discover cases by locating constructionInfo files. Try the configured root
    # first; if the mount slug differs, fall back to scanning all of /kaggle/input,
    # so we don't care what Kaggle named the mount.
    found = glob.glob(f"{CFG.INPUT_ROOT}/**/constructionInfo", recursive=True)
    if not found:
        found = glob.glob("/kaggle/input/**/constructionInfo", recursive=True)
    cidirs = sorted({os.path.dirname(p) for p in found})
    root = os.path.commonpath(cidirs) if cidirs else CFG.INPUT_ROOT
    print(f"found {len(cidirs)} case folders (root: {root})")
    manifest = []
    for cdir in cidirs:
        ci = os.path.join(cdir, "constructionInfo")
        dpath = os.path.join(cdir, "design.stl")
        if not (os.path.exists(ci) and os.path.exists(dpath)):
            continue
        teeth = parse_construction_info(ci)
        if not teeth:
            continue
        design = _load(dpath)
        dcen = design.vertices.mean(0)
        # match this folder's single design.stl to its tooth (split two-prep folders
        # share a constructionInfo listing both teeth -> pick the nearest by Center)
        tooth = min(teeth, key=lambda t: np.inf if t["center"] is None
                    else np.linalg.norm(t["center"] - dcen))
        if tooth["tmm"] is None:
            continue
        arch = fdi_arch(tooth["fdi"])
        prep_p = os.path.join(cdir, f"{arch}.stl")
        anta_p = os.path.join(cdir, f"{'lower' if arch == 'upper' else 'upper'}.stl")
        if not (os.path.exists(prep_p) and os.path.exists(anta_p)):
            continue

        tmm = tooth["tmm"]
        prep = _load(prep_p); anta = _load(anta_p)
        prep_c = (world_to_canonical(prep.vertices, tmm), prep.faces)
        anta_c = (world_to_canonical(anta.vertices, tmm), anta.faces)
        des_c  = (world_to_canonical(design.vertices, tmm), design.faces)

        if tooth["margin"] is not None and len(tooth["margin"]) >= 3:
            margin_ch = margin_channel_from_points(world_to_canonical(tooth["margin"], tmm))
        else:
            margin_ch = np.zeros((CFG.RES,) * 3, np.float32)
        x = np.stack([_voxelise(prep_c, fill=False),
                      _voxelise(anta_c, fill=False),
                      margin_ch], 0)                            # (3,R,R,R)
        y = _voxelise(des_c, fill=True)[None]                   # (1,R,R,R) solid crown
        if y.sum() < 10:                                        # voxelisation failed
            continue

        case_id = os.path.basename(cdir)
        # group key handles "-copy", "- Copy", " - Copy" so split copies never leak
        group = re.sub(r"\s*-\s*copy.*$", "", case_id, flags=re.IGNORECASE)
        np.savez_compressed(os.path.join(CFG.CACHE, f"{case_id}.npz"),
                            x=x.astype(np.float32), y=y.astype(np.float32),
                            fdi=tooth["fdi"], tmm=tmm, group=group, cdir=cdir)
        manifest.append(dict(case=case_id, fdi=tooth["fdi"], group=group,
                             cdir=cdir, crown_vox=int(y.sum())))
        print(f"  {case_id:28s} fdi {tooth['fdi']:>2}  crown vox {int(y.sum()):6d}  margin vox {int(margin_ch.sum()):5d}")
    json.dump(manifest, open(os.path.join(CFG.CACHE, "manifest.json"), "w"), indent=2)
    print(f"\ncached {len(manifest)} cases -> {CFG.CACHE}")
    return manifest

# Run once; comment out on later sessions (cache persists in /kaggle/working).
manifest = preprocess()


found 290 case folders (root: /kaggle/input/datasets/lamiaaelfadaly/single-crown/content/gdrive/MyDrive/singl crown-copy)
  12662                        fdi 16  crown vox 107701  margin vox  3589
  12856                        fdi 47  crown vox  55588  margin vox  3647
  12863                        fdi 34  crown vox  34613  margin vox  2194
  12885                        fdi 44  crown vox  52300  margin vox  2261
  12905                        fdi 47  crown vox  43424  margin vox  3023
  12915-                       fdi 14  crown vox  42843  margin vox  2023
  12929                        fdi 45  crown vox  70409  margin vox  2207
  13013                        fdi 43  crown vox  46880  margin vox  1788
  13013 - Copy                 fdi 47  crown vox  66002  margin vox  2747
  13057                        fdi 25  crown vox  32142  margin vox  2212
  13057 - Copy                 fdi 26  crown vox  62764  margin vox  2705
  13060                        fdi 17  crown vox  91127  margin 

In [10]:
# %% 4. dataset / augmentation -------------------------------------------------
import torch
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

def split_by_group(manifest, frac=CFG.SPLIT, seed=SEED):
    groups = sorted({m["group"] for m in manifest})
    rng = random.Random(seed); rng.shuffle(groups)
    n = len(groups); a = int(frac[0] * n); b = a + int(frac[1] * n)
    tr, va, te = set(groups[:a]), set(groups[a:b]), set(groups[b:])
    pick = lambda S: [m["case"] for m in manifest if m["group"] in S]
    return pick(tr), pick(va), pick(te)

class CrownVox(Dataset):
    def __init__(self, cases, train=False):
        self.cases, self.train = cases, train
    def __len__(self): return len(self.cases)
    def __getitem__(self, i):
        d = np.load(os.path.join(CFG.CACHE, f"{self.cases[i]}.npz"))
        x, y = d["x"].copy(), d["y"].copy()
        if self.train:                      # safe augmentation: rotate about insertion axis only
            k = random.randint(0, 3)
            x = np.rot90(x, k, axes=(1, 2)); y = np.rot90(y, k, axes=(1, 2))
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)))


In [11]:
# %% 5. 3D U-Net ---------------------------------------------------------------
import torch.nn as nn
import torch.nn.functional as F

def conv_block(ci, co):
    return nn.Sequential(
        nn.Conv3d(ci, co, 3, padding=1, bias=False), nn.GroupNorm(8, co), nn.ReLU(inplace=True),
        nn.Conv3d(co, co, 3, padding=1, bias=False), nn.GroupNorm(8, co), nn.ReLU(inplace=True))

class UNet3D(nn.Module):
    def __init__(self, in_ch=CFG.IN_CH, base=CFG.BASE_CH):
        super().__init__()
        b = base
        self.e1 = conv_block(in_ch, b);     self.e2 = conv_block(b, b * 2)
        self.e3 = conv_block(b * 2, b * 4)
        self.bott = conv_block(b * 4, b * 8)
        self.pool = nn.MaxPool3d(2)
        self.up3 = nn.ConvTranspose3d(b * 8, b * 4, 2, 2); self.d3 = conv_block(b * 8, b * 4)
        self.up2 = nn.ConvTranspose3d(b * 4, b * 2, 2, 2); self.d2 = conv_block(b * 4, b * 2)
        self.up1 = nn.ConvTranspose3d(b * 2, b, 2, 2);     self.d1 = conv_block(b * 2, b)
        self.head = nn.Conv3d(b, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        z = self.bott(self.pool(e3))
        d = self.d3(torch.cat([self.up3(z), e3], 1))
        d = self.d2(torch.cat([self.up2(d), e2], 1))
        d = self.d1(torch.cat([self.up1(d), e1], 1))
        return self.head(d)                 # logits (B,1,R,R,R)


In [12]:
# %% 6. train ------------------------------------------------------------------
def dice_bce(logits, y, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((2, 3, 4)); den = p.sum((2, 3, 4)) + y.sum((2, 3, 4))
    dice = 1 - ((2 * inter + eps) / (den + eps)).mean()
    return dice + F.binary_cross_entropy_with_logits(logits, y)

@torch.no_grad()
def iou(logits, y, thr=0.5):
    p = (torch.sigmoid(logits) > thr).float()
    i = (p * y).sum((2, 3, 4)); u = ((p + y) >= 1).float().sum((2, 3, 4))
    return ((i + 1) / (u + 1)).mean().item()

def train():
    tr, va, te = split_by_group(manifest)
    json.dump(dict(train=tr, val=va, test=te), open(f"{CFG.OUT}/splits.json", "w"), indent=2)
    print(f"train {len(tr)}  val {len(va)}  test {len(te)}")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    dl_tr = DataLoader(CrownVox(tr, True), CFG.BATCH, shuffle=True, num_workers=2, drop_last=True)
    dl_va = DataLoader(CrownVox(va), CFG.BATCH)
    net = UNet3D().to(dev)
    opt = torch.optim.Adam(net.parameters(), CFG.LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG.EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP)
    best = -1
    for ep in range(CFG.EPOCHS):
        net.train()
        for x, y in dl_tr:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=CFG.AMP):
                loss = dice_bce(net(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        net.eval(); vs = []
        with torch.no_grad():
            for x, y in dl_va:
                x, y = x.to(dev), y.to(dev)
                vs.append(iou(net(x), y))
        v = float(np.mean(vs)) if vs else 0.0
        print(f"epoch {ep:3d}  val IoU {v:.4f}")
        if v > best:
            best = v; torch.save(net.state_dict(), f"{CFG.CKPT}/best.pt")
    print(f"best val IoU {best:.4f}  -> {CFG.CKPT}/best.pt")
    return te

test_cases = train()


train 53  val 8  test 29


/tmp/ipykernel_58/1676965836.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP)
/tmp/ipykernel_58/1676965836.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):


epoch   0  val IoU 0.0336
epoch   1  val IoU 0.3956
epoch   2  val IoU 0.4693
epoch   3  val IoU 0.5121
epoch   4  val IoU 0.4059
epoch   5  val IoU 0.5076
epoch   6  val IoU 0.5352
epoch   7  val IoU 0.5227
epoch   8  val IoU 0.5094
epoch   9  val IoU 0.5232
epoch  10  val IoU 0.4573
epoch  11  val IoU 0.5495
epoch  12  val IoU 0.5524
epoch  13  val IoU 0.5339
epoch  14  val IoU 0.5482
epoch  15  val IoU 0.5260
epoch  16  val IoU 0.5177
epoch  17  val IoU 0.5332
epoch  18  val IoU 0.5439
epoch  19  val IoU 0.5606
epoch  20  val IoU 0.5279
epoch  21  val IoU 0.5205
epoch  22  val IoU 0.5639
epoch  23  val IoU 0.5403
epoch  24  val IoU 0.5592
epoch  25  val IoU 0.5631
epoch  26  val IoU 0.5499
epoch  27  val IoU 0.5720
epoch  28  val IoU 0.5707
epoch  29  val IoU 0.5808
epoch  30  val IoU 0.5747
epoch  31  val IoU 0.5658
epoch  32  val IoU 0.5786
epoch  33  val IoU 0.5800
epoch  34  val IoU 0.5590
epoch  35  val IoU 0.5389
epoch  36  val IoU 0.5838
epoch  37  val IoU 0.5711
epoch  38  v

In [13]:
# %% 7. inference + mesh -------------------------------------------------------
from skimage import measure

@torch.no_grad()
def predict_mesh(net, case, dev):
    d = np.load(os.path.join(CFG.CACHE, f"{case}.npz"))
    x = torch.from_numpy(d["x"][None]).to(dev)
    prob = torch.sigmoid(net(x))[0, 0].cpu().numpy()
    pred = (prob > getattr(CFG, 'THR', 0.5)).astype(np.float32)
    # keep the largest connected component, then surface it
    lbl = measure.label(pred)
    if lbl.max() > 0:
        pred = (lbl == np.argmax(np.bincount(lbl.flat)[1:]) + 1).astype(np.float32)
    return pred, d["y"][0], d["tmm"]


def crown_world_mesh(vox, tmm):
    """Marching-cubes the predicted occupancy and place it back in world coords
    (voxel idx -> canonical mm -> world via forward ToothModelMatrix). Returns a
    trimesh that overlays the original scans / design.stl directly, or None."""
    if vox.sum() < 4:
        return None
    verts, faces, *_ = measure.marching_cubes(vox, level=0.5)
    verts_mm = (verts - CFG.RES // 2) * CFG.PITCH
    world = (np.c_[verts_mm, np.ones(len(verts_mm))] @ tmm)[:, :3]
    return trimesh.Trimesh(world, faces, process=False)


# --- deterministic margin-snap -------------------------------------------------
# Trim the predicted crown exactly at the preparation finish line so its cervical
# edge coincides with the real margin polyline. Done in the canonical frame
# (insertion axis = +Z), by subtracting a solid "plug" swept apically from the
# margin: crown - plug removes any flash apical to the finish line and cuts the
# crown along the true (3D, Z-varying) margin curve. Robust boolean via manifold3d;
# falls back to the raw crown if the boolean can't run on a given case.

def _margin_plug(margin_canon, depth=8.0):
    """Closed solid spanning from the margin curve `depth` mm apically (-Z)."""
    M = np.asarray(margin_canon, float)
    n = len(M)
    Mlow = M - np.array([0, 0, depth])
    V = np.vstack([M, Mlow, M.mean(0), Mlow.mean(0)])     # +2 cap centroids
    top_c, low_c = 2 * n, 2 * n + 1
    F = []
    for i in range(n):
        j = (i + 1) % n
        F += [[i, j, n + j], [i, n + j, n + i]]           # side wall
        F += [[top_c, j, i]]                              # top cap (margin)
        F += [[low_c, n + i, n + j]]                      # bottom cap
    return trimesh.Trimesh(np.asarray(V), np.asarray(F), process=True)

def snap_to_margin(crown_world, margin_world, tmm):
    """Return the crown trimmed to the finish line (world coords), or the raw
    crown if anything fails. Never raises."""
    if margin_world is None or len(margin_world) < 8:
        return crown_world
    try:
        cc = crown_world.copy()
        cc.vertices = world_to_canonical(cc.vertices, tmm)
        plug = _margin_plug(world_to_canonical(margin_world, tmm))
        cut = trimesh.boolean.difference([cc, plug], engine="manifold")
        if cut is None or cut.is_empty or len(cut.vertices) < 10:
            return crown_world
        cut.vertices = (np.c_[cut.vertices, np.ones(len(cut.vertices))] @ tmm)[:, :3]
        return cut
    except Exception:
        return crown_world


In [14]:
# %% 8. clinical metrics: deviation heatmap + occlusal + margin -----------------
from scipy.spatial import cKDTree
import matplotlib.cm as _cm
import csv

def _sample(mesh, n=20000):
    pts, _ = trimesh.sample.sample_surface(mesh, n)
    return np.asarray(pts)

def _cmap(vals, vmin, vmax):
    t = np.clip((np.asarray(vals) - vmin) / (vmax - vmin + 1e-9), 0, 1)
    return (_cm.coolwarm(t)[:, :3] * 255).astype(np.uint8)

def vox_iou(pred, gt):
    i = (pred * gt).sum(); u = ((pred + gt) >= 1).sum()
    return float((i + 1) / (u + 1))

def surface_metrics(ai, design, n=20000):
    """Symmetric Hausdorff + RMS surface distance (mm) between two world meshes."""
    P, G = _sample(ai, n), _sample(design, n)
    dPG = cKDTree(G).query(P)[0]; dGP = cKDTree(P).query(G)[0]
    hd = max(dPG.max(), dGP.max())
    rms = math.sqrt((np.r_[dPG, dGP] ** 2).mean())
    return hd, rms

def deviation_heatmap(ai, design, out_ply, clip=0.5):
    """Per-vertex signed deviation of the AI crown from the exocad design.
    + = AI proud/over-contoured, - = deficient. Writes a colour-mapped PLY
    (open in MeshLab/CloudCompare) and returns (mean|dev|, %within TOL, %within 0.2mm)."""
    try:
        sd = trimesh.proximity.ProximityQuery(design).signed_distance(ai.vertices)
        dev = -np.asarray(sd)
    except Exception:                                   # design not watertight -> unsigned
        dev = cKDTree(_sample(design, 60000)).query(ai.vertices)[0]
    rgb = _cmap(dev, -clip, clip)
    m = ai.copy(); m.visual.vertex_colors = np.c_[rgb, np.full(len(rgb), 255, np.uint8)]
    m.export(out_ply)
    a = np.abs(dev)
    return float(a.mean()), float((a <= CFG.TOL_MM).mean() * 100), float((a <= 0.2).mean() * 100)

def occlusal_gap(crown, anta, c0):
    """Nearest distance from the crown surface to the opposing (antagonist) arch.
    Returns the minimum gap (occlusal clearance/contact, mm). Restricts the
    antagonist to the local ROI so far-away arch points don't pollute the query."""
    apts = _sample(anta, 60000)
    apts = apts[np.linalg.norm(apts - c0, axis=1) < CFG.ROI_MM]
    if len(apts) < 10:
        return float("nan")
    d = cKDTree(apts).query(_sample(crown, 20000))[0]
    return float(d.min())

def margin_discrepancy(crown, margin_pts):
    """Distance from the true preparation finish line (constructionInfo Margin,
    world coords) to the crown surface -> marginal discrepancy (mm)."""
    if margin_pts is None or len(margin_pts) < 3:
        return float("nan"), float("nan")
    d = cKDTree(_sample(crown, 40000)).query(margin_pts)[0]
    return float(d.mean()), float(d.max())

def _case_world_refs(case):
    """Reload the case's world-space design crown, antagonist arch and margin
    polyline (cheap; the raw scans are already in one world frame)."""
    cdir = str(np.load(os.path.join(CFG.CACHE, f"{case}.npz"))["cdir"])
    teeth = parse_construction_info(os.path.join(cdir, "constructionInfo"))
    design = _load(os.path.join(cdir, "design.stl"))
    dcen = design.vertices.mean(0)
    tooth = min(teeth, key=lambda t: np.inf if t["center"] is None
                else np.linalg.norm(t["center"] - dcen))
    arch = fdi_arch(tooth["fdi"])
    anta = _load(os.path.join(cdir, f"{'lower' if arch == 'upper' else 'upper'}.stl"))
    return design, anta, tooth["margin"], tooth["fdi"]


In [15]:
# %% 9. evaluate -> metrics.csv + STL + heatmaps -------------------------------
FIELDS = ["case", "fdi", "IoU", "Hausdorff_mm", "RMS_mm",
          "occlusal_gap_AI_mm", "occlusal_gap_design_mm",
          "margin_mean_mm", "margin_max_mm", "margin_fit_max_mm", "dev_within_tol_pct"]

def evaluate(test_cases):
    dev_ = "cuda" if torch.cuda.is_available() else "cpu"
    net = UNet3D().to(dev_); net.load_state_dict(torch.load(f"{CFG.CKPT}/best.pt", map_location=dev_)); net.eval()
    stl_dir = os.path.join(CFG.OUT, "stl"); heat_dir = os.path.join(CFG.OUT, "heatmap")
    os.makedirs(stl_dir, exist_ok=True); os.makedirs(heat_dir, exist_ok=True)
    rows = []
    for c in test_cases:
        pred, gt, tmm = predict_mesh(net, c, dev_)
        ai = crown_world_mesh(pred, tmm)
        if ai is None:
            continue
        ai.export(os.path.join(stl_dir, f"{c}_AI_crown.stl"))
        design, anta, margin, fdi = _case_world_refs(c)
        c0 = np.asarray(ai.vertices).mean(0)
        hd, rms = surface_metrics(ai, design)
        og_ai = occlusal_gap(ai, anta, c0)
        og_de = occlusal_gap(design, anta, c0)            # exocad's occlusal gap = reference
        mg_mean, mg_max = margin_discrepancy(ai, margin)  # model's raw marginal accuracy
        fitted = snap_to_margin(ai, margin, tmm)          # seatable crown
        fitted.export(os.path.join(stl_dir, f"{c}_AI_crown_fitted.stl"))
        _, mg_fit_max = margin_discrepancy(fitted, margin)   # seating residual (QC, ~0)
        _, pct_tol, _ = deviation_heatmap(ai, design, os.path.join(heat_dir, f"{c}_dev.ply"))
        rows.append(dict(case=c, fdi=fdi, IoU=round(vox_iou(pred, gt), 4),
                         Hausdorff_mm=round(hd, 4), RMS_mm=round(rms, 4),
                         occlusal_gap_AI_mm=round(og_ai, 4), occlusal_gap_design_mm=round(og_de, 4),
                         margin_mean_mm=round(mg_mean, 4), margin_max_mm=round(mg_max, 4),
                         margin_fit_max_mm=round(mg_fit_max, 4),
                         dev_within_tol_pct=round(pct_tol, 1)))
        print(f"  {c:26s} IoU {rows[-1]['IoU']:.3f} HD {hd:.2f} RMS {rms:.2f} "
              f"occl(AI/exo {og_ai:.2f}/{og_de:.2f}) margin {mg_mean:.2f}/{mg_max:.2f} "
              f"within{CFG.TOL_MM} {pct_tol:.0f}%")
    with open(f"{CFG.OUT}/metrics.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS); w.writeheader(); w.writerows(rows)

    print("\n=== test summary (metrics.csv -> MedCalc) ===")
    for k in FIELDS[2:]:
        col = np.array([r[k] for r in rows], float); col = col[~np.isnan(col)]
        if len(col):
            print(f"  {k:22s} mean {col.mean():.4f}  RMS {math.sqrt((col**2).mean()):.4f}  SD {col.std(ddof=1):.4f}")
    occl_acc = np.abs(np.array([r["occlusal_gap_AI_mm"] - r["occlusal_gap_design_mm"] for r in rows], float))
    occl_acc = occl_acc[~np.isnan(occl_acc)]
    if len(occl_acc):
        print(f"  occlusal accuracy (|AI-exocad gap|): mean {occl_acc.mean():.4f} mm")
    print(f"\n  equivalence margin for write-up: RMS / margin deviation <= {CFG.TOL_MM} mm")
    print(f"  STL crowns : {stl_dir}/   (<case>_AI_crown.stl = morphology; "
          f"<case>_AI_crown_fitted.stl = margin-snapped, seatable)")
    print(f"  heatmaps   : {heat_dir}/   (PLY, signed deviation; open in MeshLab/CloudCompare)")
    print("  READING THE MARGIN COLUMNS:")
    print("   - margin_mean/max_mm  = the MODEL's raw marginal accuracy (report this;")
    print(f"     floored by voxel pitch {CFG.PITCH:.3f} mm, so use RES>=128 for the final run).")
    print("   - margin_fit_max_mm   = residual AFTER margin-snap (~0 = the delivered crown")
    print("     seats on the finish line). The _fitted.stl is the clinically usable crown.")
    print("  Occlusal & morphology metrics are likewise sharpened by RES>=128; "
          "prove the pipeline at 64, report at 128.")
    return rows

metrics = evaluate(test_cases)


  12863                      IoU 0.503 HD 2.35 RMS 0.58 occl(AI/exo 0.01/0.02) margin 0.08/0.15 within0.1 25%
  13013                      IoU 0.625 HD 1.98 RMS 0.56 occl(AI/exo 0.86/0.31) margin 0.08/0.18 within0.1 18%
  13013 - Copy               IoU 0.670 HD 1.13 RMS 0.36 occl(AI/exo 2.01/1.62) margin 0.08/0.15 within0.1 22%
  13057                      IoU 0.585 HD 1.58 RMS 0.48 occl(AI/exo 0.01/0.02) margin 0.10/0.20 within0.1 11%
  13057 - Copy               IoU 0.507 HD 2.03 RMS 0.56 occl(AI/exo 0.02/0.01) margin 0.10/0.19 within0.1 22%
  18264                      IoU 0.686 HD 1.46 RMS 0.36 occl(AI/exo 0.01/0.01) margin 0.08/0.15 within0.1 20%
  19099                      IoU 0.713 HD 1.22 RMS 0.37 occl(AI/exo 0.04/0.01) margin 0.09/0.16 within0.1 18%
  20094                      IoU 0.742 HD 1.16 RMS 0.34 occl(AI/exo 0.21/0.11) margin 0.08/0.15 within0.1 28%
  2024-03-25_00333-001       IoU 0.479 HD 5.38 RMS 1.19 occl(AI/exo nan/nan) margin 0.08/0.16 within0.1 14%
  2025-09-21

In [16]:
# %% DEMO — design a crown for a held-out preparation (Stage 1) -----------------
import os, numpy as np, trimesh, torch
DEMO = "/kaggle/working/demo"; os.makedirs(DEMO, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
_tr, _va, _te = split_by_group(manifest)

def design_crown(case, finish=True):
    """Run a single (unseen) prep through Stage 1 -> crown. Exports the AI crown,
    the exocad design (for comparison), and a deviation heatmap; prints accuracy."""
    net = UNet3D(in_ch=CFG.IN_CH).to(DEV)
    net.load_state_dict(torch.load(f"{CFG.CKPT}/best.pt", map_location=DEV)); net.eval()

    pred, gt, tmm = predict_mesh(net, case, DEV)        # prep+antagonist+margin -> crown voxels
    ai = crown_world_mesh(pred, tmm)                    # -> world-space mesh
    design, anta, margin, fdi = _case_world_refs(case)

    ai.export(os.path.join(DEMO, f"{case}_AI_crown.stl"))
    design.export(os.path.join(DEMO, f"{case}_exocad_design.stl"))
    if finish:                                          # optional: seat on the margin
        snap_to_margin(ai, margin, tmm).export(os.path.join(DEMO, f"{case}_AI_crown_fitted.stl"))
    dmean, pct01, pct02 = deviation_heatmap(ai, design, os.path.join(DEMO, f"{case}_dev.ply"))
    hd, rms = surface_metrics(ai, design)
    mg_mean, mg_max = margin_discrepancy(ai, margin)

    print(f"case {case}  (tooth {fdi})  — UNSEEN by the model")
    print(f"  IoU        {vox_iou(pred, gt):.3f}")
    print(f"  RMS        {rms:.3f} mm   Hausdorff {hd:.3f} mm")
    print(f"  margin     {mg_mean:.3f} mm (mean)  {mg_max:.3f} mm (max)")
    print(f"  surface within 0.2 mm: {pct02:.0f}%")
    print(f"  files -> {DEMO}/  (AI_crown.stl, exocad_design.stl, dev.ply)")
    return ai

# demo on a held-out TEST case (the model never saw it during training):
design_crown(_te[0])

case 12863  (tooth 34)  — UNSEEN by the model
  IoU        0.503
  RMS        0.577 mm   Hausdorff 2.341 mm
  margin     0.079 mm (mean)  0.151 mm (max)
  surface within 0.2 mm: 40%
  files -> /kaggle/working/demo/  (AI_crown.stl, exocad_design.stl, dev.ply)


<trimesh.Trimesh(vertices.shape=(14952, 3), faces.shape=(29900, 3))>

In [17]:
# FINISHING — turn Stage-1 crowns into watertight, millable SHELLS
# Append after Stage 1. For each case it: predicts the crown solid, carves an
# intaglio from the prepared die (offset by a cement gap), snaps the margin,
# smooths, and exports a watertight/manifold STL + a QC log.
#
# Requires the prep arch (upper.stl/lower.stl) -> works on your 91 COMPLETE cases.
# NOTE: cement gap is limited by voxel pitch (~0.156 mm), so these seat loosely —
# good for a MILL/MATERIAL TEST, not clinical seating. Not a certified restoration

In [18]:
# %% F0 — finishing config -----------------------------------------------------
class FIN:
    OUT        = "/kaggle/working/out_mill"
    HEAT       = "/kaggle/working/out_mill/fit_heatmap"
    CEMENT_MM  = 0.08
    MIN_WALL   = 0.50      # material minimum wall (zirconia ~0.5) — QC flag
    SMOOTH_IT  = 10
import os, glob, numpy as np, trimesh, torch
from scipy.ndimage import binary_dilation, label as cc_label, distance_transform_edt
from scipy.spatial import cKDTree
from skimage import measure
os.makedirs(FIN.OUT, exist_ok=True); os.makedirs(FIN.HEAT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def find_arch_file(folder, arch):
    tgt = f"{arch}.stl".lower()
    for f in glob.glob(os.path.join(folder, "*.stl")):
        if os.path.basename(f).lower() == tgt:
            return f
    return None

In [19]:
# %% F1 — carve + smooth + clean + snap-last + QC (+ mill_ready flag) -----------
FIN.MARGIN_MAX = 0.7      # margin (mm) a shell must beat to be called mill-ready

def _prep_die_occ(prep_canon_verts, prep_faces):
    shell = _voxelise((prep_canon_verts, prep_faces), fill=False) > 0
    anyk = shell.any(2)
    kmax = shell.shape[2] - 1 - np.argmax(shell[:, :, ::-1], axis=2)
    kmax = np.where(anyk, kmax, -1)
    kgrid = np.arange(shell.shape[2])[None, None, :]
    return ((kgrid <= kmax[:, :, None]) & anyk[:, :, None])

def _wall_thickness_mm(shell_occ):
    edt = distance_transform_edt(shell_occ) * CFG.PITCH * 2.0
    w = edt[shell_occ]
    if w.size == 0:
        return float("nan"), float("nan")
    return float(np.percentile(w, 2)), float(np.median(w))

def _fit_heatmap(shell_world, prep_world, c0, out_ply, clip=0.5):
    pv_pts, _ = trimesh.sample.sample_surface(prep_world, 60000)
    pv_pts = np.asarray(pv_pts)
    pv_pts = pv_pts[np.linalg.norm(pv_pts - c0, axis=1) < CFG.ROI_MM]
    if len(pv_pts) < 10:
        return
    d = cKDTree(pv_pts).query(shell_world.vertices)[0]
    rgb = _cmap(d, 0.0, clip)
    m = shell_world.copy()
    m.visual.vertex_colors = np.c_[rgb, np.full(len(rgb), 255, np.uint8)]
    m.export(out_ply)

def finish_case(net, case):
    pred, _, tmm = predict_mesh(net, case, DEV)
    d = np.load(os.path.join(CFG.CACHE, f"{case}.npz"))
    cdir = str(d["cdir"]); fdi = int(d["fdi"])
    prep_p = find_arch_file(cdir, fdi_arch(fdi))
    if prep_p is None:
        return None, {"case": case, "ok": False, "reason": "no prep arch"}
    prep = _load(prep_p)
    pv = world_to_canonical(prep.vertices, tmm)
    die = _prep_die_occ(pv, prep.faces)
    if die.sum() == 0:
        return None, {"case": case, "ok": False, "reason": "intaglio_voxels=0 (bulk/bad scan)"}
    gap_vox = max(1, int(round(FIN.CEMENT_MM / CFG.PITCH)))
    die = binary_dilation(die, iterations=gap_vox)

    shell = (pred > 0.5) & (~die)
    if shell.sum() < 50:
        return None, {"case": case, "ok": False, "reason": "shell empty after carve"}
    lab, n = cc_label(shell)
    if n > 1:
        shell = lab == (np.argmax(np.bincount(lab.flat)[1:]) + 1)
    wmin, wmed = _wall_thickness_mm(shell)

    verts, faces, *_ = measure.marching_cubes(shell.astype(np.float32), level=0.5)
    verts = (verts - CFG.RES // 2) * CFG.PITCH
    world = (np.c_[verts, np.ones(len(verts))] @ tmm)[:, :3]
    m = trimesh.Trimesh(world, faces, process=True)

    refs = _case_world_refs(case)                   # (design, anta, margin, fdi)
    margin = refs[2]
    try:
        trimesh.smoothing.filter_taubin(m, lamb=0.5, nu=-0.53, iterations=FIN.SMOOTH_IT)
    except Exception:
        pass
    m.merge_vertices(); m.fill_holes(); m.fix_normals()
    m = snap_to_margin(m, margin, tmm)              # final shaping step

    mg_mean, mg_max = margin_discrepancy(m, margin)
    c0 = np.asarray(m.vertices).mean(0)
    _fit_heatmap(m, prep, c0, os.path.join(FIN.HEAT, f"{case}_fit.ply"))

    out = os.path.join(FIN.OUT, f"{case}_AI_crown_SHELL.stl")
    m.export(out)
    watertight = bool(m.is_watertight)
    mill_ready = bool(watertight and mg_mean <= FIN.MARGIN_MAX)
    return m, {"case": case, "ok": True,
               "watertight": watertight,
               "mill_ready": mill_ready,
               "n_faces": int(len(m.faces)),
               "volume_mm3": round(float(m.volume), 1) if watertight else None,
               "wall_min_mm": round(wmin, 3), "wall_med_mm": round(wmed, 3),
               "margin_mean_mm": round(mg_mean, 3), "margin_max_mm": round(mg_max, 3),
               "cement_gap_mm": round(gap_vox * CFG.PITCH, 3),
               "file": out}

In [20]:
# %% F2 — run finishing + QC summary (matches new F1) --------------------------
def finish_crowns(cases):
    net = UNet3D(in_ch=CFG.IN_CH).to(DEV)
    net.load_state_dict(torch.load(f"{CFG.CKPT}/best.pt", map_location=DEV)); net.eval()
    log = []
    for c in cases:
        try:
            _, qc = finish_case(net, c)
        except Exception as e:
            qc = {"case": c, "ok": False, "reason": repr(e)[:80]}
        log.append(qc)
        if qc.get("ok"):
            tag = "  MILL-READY" if qc["mill_ready"] else "  hold"
            print(f"  {c:26s} wt={qc['watertight']} wall {qc['wall_med_mm']:.2f}mm "
                  f"margin {qc['margin_mean_mm']:.2f}/{qc['margin_max_mm']:.2f}mm "
                  f"vol {qc['volume_mm3']}{tag}")
        else:
            print(f"  {c:26s} EXCLUDED: {qc.get('reason')}")
    import csv
    keys = sorted({k for q in log for k in q})
    with open(os.path.join(FIN.OUT, "mill_qc.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(log)
    ready = [q for q in log if q.get("ok") and q["mill_ready"]]
    print(f"\n{len([q for q in log if q.get('ok')])} shells produced | "
          f"{len(ready)} MILL-READY (watertight + margin <= {FIN.MARGIN_MAX} mm)")
    print("mill-ready cases:", sorted(q["case"] for q in ready))
    print(f"STLs: {FIN.OUT}/   fit heatmaps: {FIN.HEAT}/   QC: mill_qc.csv")
    return log

DEV = "cuda" if torch.cuda.is_available() else "cpu"
_tr, _va, _te = split_by_group(manifest)
log = finish_crowns(_te)        # whole test set; capture the log

  12863                      wt=True wall 0.88mm margin 0.36/0.80mm vol 148.4  MILL-READY
  13013                      wt=True wall 0.88mm margin 0.42/1.03mm vol 166.8  MILL-READY
  13013 - Copy               wt=True wall 0.70mm margin 0.48/1.67mm vol 141.4  MILL-READY
  13057                      wt=True wall 0.62mm margin 2.36/3.41mm vol 34.5  hold
  13057 - Copy               wt=True wall 0.62mm margin 2.93/4.42mm vol 55.6  hold
  18264                      wt=True wall 0.62mm margin 2.59/4.67mm vol 34.0  hold
  19099                      wt=True wall 0.70mm margin 0.42/1.21mm vol 205.6  MILL-READY
  20094                      wt=True wall 0.77mm margin 0.45/1.67mm vol 190.9  MILL-READY
  2024-03-25_00333-001       EXCLUDED: intaglio_voxels=0 (bulk/bad scan)
  2025-09-21_98772-002       wt=True wall 0.88mm margin 2.81/4.62mm vol 89.2  hold
  2025-09-21_98776-001       wt=True wall 0.88mm margin 3.02/4.52mm vol 87.7  hold
  2025-09-28_98771-003       wt=True wall 0.70mm margin 0.62/1

In [21]:
# %% S0 — stage-2 config -------------------------------------------------------
class S2:
    CACHE   = "/kaggle/working/stage2cache"      # env + design point clouds + templates
    CKPT    = "/kaggle/working/ckpt2"
    OUT     = "/kaggle/working/out_stage2"
    LATENT  = 256
    N_CHAMF = 2500          # points used per Chamfer term
    MAXDISP = 6.0           # mm, bound on per-vertex offset (was 4.0 — allow bigger teeth)
    LAMBDA_LAP = 0.5        # smoothness weight (was 5.0 — let the mesh actually deform)
    EPOCHS  = 250
    LR      = 3e-4          # steadier than 1e-3 (less wandering)
    ENC_CH  = 4             # [prep, antagonist, margin, stage1-envelope]
for d in (S2.CACHE, S2.CKPT, S2.OUT):
    os.makedirs(d, exist_ok=True)

import torch, torch.nn as nn, torch.nn.functional as F
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# recover the same split Stage 1 used (no leakage: copies grouped together)
_tr, _va, _te = split_by_group(manifest)
print(f"stage-2 split  train {len(_tr)}  val {len(_va)}  test {len(_te)}")
print(f"tuning -> LAMBDA_LAP {S2.LAMBDA_LAP}  MAXDISP {S2.MAXDISP}  LR {S2.LR}  EPOCHS {S2.EPOCHS}")

stage-2 split  train 53  val 8  test 29
tuning -> LAMBDA_LAP 0.5  MAXDISP 6.0  LR 0.0003  EPOCHS 250


In [22]:
# %% S1 — per-FDI anatomical templates (from TRAIN design crowns) ---------------
# One crisp real crown per tooth type, in canonical frame, used as the shape to
# deform. Drawn only from training cases so test crowns never seed their own template.

def _canon_design(case):
    d = np.load(os.path.join(CFG.CACHE, f"{case}.npz"))
    refs = _case_world_refs(case)          # (design_world, anta, margin, fdi)
    design = refs[0]
    V = world_to_canonical(design.vertices, d["tmm"])
    return trimesh.Trimesh(V, design.faces, process=False), int(d["fdi"])

def build_templates(train_cases):
    by_fdi = {}
    for c in train_cases:
        try:
            m, fdi = _canon_design(c)
        except Exception:
            continue
        # prefer a median-sized, clean exemplar per FDI
        by_fdi.setdefault(fdi, []).append((len(m.vertices), c, m))
    templates = {}
    for fdi, lst in by_fdi.items():
        lst.sort(key=lambda t: t[0])
        _, case, m = lst[len(lst) // 2]               # median vertex count
        # build edge list once for the Laplacian smoothness term
        e = m.edges_unique
        templates[fdi] = dict(verts=np.asarray(m.vertices, np.float32),
                              faces=np.asarray(m.faces, np.int64),
                              edges=np.asarray(e, np.int64), src=case)
        print(f"  fdi {fdi:>2}: template from {case}  ({len(m.vertices)} verts)")
    np.save(os.path.join(S2.CACHE, "templates.npy"), templates, allow_pickle=True)
    return templates

templates = build_templates(_tr)
# fallback for any FDI that only appears in val/test
_all_fdi = {int(np.load(os.path.join(CFG.CACHE, f'{c}.npz'))['fdi']) for c in _tr + _va + _te}
for fdi in _all_fdi:
    if fdi not in templates:
        anyf = next(iter(templates)); templates[fdi] = templates[anyf]
        print(f"  fdi {fdi}: no train exemplar -> reuse fdi {anyf}")


  fdi 16: template from 12662  (115530 verts)
  fdi 47: template from 21281  (107526 verts)
  fdi 44: template from 21329  (85374 verts)
  fdi 14: template from 12915-  (96036 verts)
  fdi 45: template from 21080  (105186 verts)
  fdi 17: template from 13060  (118860 verts)
  fdi 15: template from 18661  (104634 verts)
  fdi 36: template from 19785  (147510 verts)
  fdi 26: template from 2025-10-20_98795-008  (147336 verts)
  fdi 24: template from 18699  (106020 verts)
  fdi 25: template from 19932  (96000 verts)
  fdi 11: template from 20212  (102384 verts)
  fdi 37: template from 2024-09-29_00315-002  (200196 verts)
  fdi 35: template from 20264  (106656 verts)
  fdi 46: template from 2025-11-18_00224-001  (139716 verts)
  fdi 22: template from 2025-10-16_98795-003  (68202 verts)
  fdi 34: template from 21246  (103464 verts)
  fdi 43: no train exemplar -> reuse fdi 16


In [23]:
# %% S2 — precompute Stage-1 envelope + design point clouds per case -----------
# Stage-1 envelope is the SOFT predicted occupancy (so Stage 2 reads Stage 1's
# answer, identically at train and test time). design point cloud = Chamfer target.

@torch.no_grad()
def precompute(cases):
    net1 = UNet3D(in_ch=CFG.IN_CH).to(DEV)
    net1.load_state_dict(torch.load(f"{CFG.CKPT}/best.pt", map_location=DEV)); net1.eval()
    for c in cases:
        d = np.load(os.path.join(CFG.CACHE, f"{c}.npz"))
        x = torch.from_numpy(d["x"][None]).to(DEV)
        env = torch.sigmoid(net1(x))[0, 0].cpu().numpy().astype(np.float32)   # (R,R,R)
        # design points in canonical frame
        m, _ = _canon_design(c)
        pts = _sample(m, 8000).astype(np.float32)
        np.savez_compressed(os.path.join(S2.CACHE, f"{c}.npz"), env=env, dpts=pts)
    print(f"precomputed env + design clouds for {len(cases)} cases")

precompute(_tr + _va + _te)


precomputed env + design clouds for 90 cases


In [24]:
# %% S3 — stage-2 model + losses ----------------------------------------------
def enc_block(ci, co):
    return nn.Sequential(nn.Conv3d(ci, co, 3, 2, 1, bias=False),
                         nn.GroupNorm(8, co), nn.ReLU(inplace=True))

class DeformNet(nn.Module):
    """3D-CNN context encoder -> latent; MLP maps (template vertex, latent)->offset."""
    def __init__(self, enc_ch=S2.ENC_CH, latent=S2.LATENT, base=16):
        super().__init__()
        self.enc = nn.Sequential(
            enc_block(enc_ch, base), enc_block(base, base * 2),
            enc_block(base * 2, base * 4), enc_block(base * 4, base * 4),
            nn.AdaptiveAvgPool3d(1))
        self.fc = nn.Linear(base * 4, latent)
        self.mlp = nn.Sequential(
            nn.Linear(3 + latent, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 3))
    def forward(self, grid, tverts):
        z = self.fc(self.enc(grid).flatten(1))             # (1, latent)
        v = tverts / (CFG.ROI_MM / 2)                      # normalise verts ~[-1,1]
        zc = z.expand(tverts.shape[0], -1)
        off = torch.tanh(self.mlp(torch.cat([v, zc], 1))) * S2.MAXDISP
        return tverts + off, off                           # deformed verts, offsets (mm)

def chamfer(P, G):
    D = torch.cdist(P, G)                                   # (Np,Ng)
    return D.min(1).values.mean() + D.min(0).values.mean()

def lap_smooth(off, edges):
    return ((off[edges[:, 0]] - off[edges[:, 1]]) ** 2).sum(1).mean()

def _grid4(case):
    """[prep, antagonist, margin, stage1-env] -> (1,4,R,R,R) tensor."""
    x = np.load(os.path.join(CFG.CACHE, f"{case}.npz"))["x"]      # (3,R,R,R)
    env = np.load(os.path.join(S2.CACHE, f"{case}.npz"))["env"][None]
    g = np.concatenate([x, env], 0)[None]
    return torch.from_numpy(g.astype(np.float32))

def _tmpl(case):
    fdi = int(np.load(os.path.join(CFG.CACHE, f"{case}.npz"))["fdi"])
    t = templates[fdi]
    return (torch.from_numpy(t["verts"]).to(DEV),
            torch.from_numpy(t["edges"]).to(DEV), t["faces"])


In [25]:
# %% S4 — SELF-CHECK: overfit a few TRAINING cases, export anatomical crowns ----
def sanity_overfit(case, steps=250):
    net = DeformNet().to(DEV)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    g = _grid4(case).to(DEV)
    tv, ed, faces = _tmpl(case)
    G = torch.from_numpy(np.load(os.path.join(S2.CACHE, f"{case}.npz"))["dpts"]).to(DEV)
    last = None
    for s in range(steps):
        opt.zero_grad()
        P, off = net(g, tv)
        Pi = P[torch.randint(len(P), (S2.N_CHAMF,), device=DEV)]
        Gi = G[torch.randint(len(G), (S2.N_CHAMF,), device=DEV)]
        loss = chamfer(Pi, Gi) + S2.LAMBDA_LAP * lap_smooth(off, ed)
        loss.backward(); opt.step()
        last = loss.item()
    with torch.no_grad():
        V = net(g, tv)[0].cpu().numpy()
    d = np.load(os.path.join(CFG.CACHE, f"{case}.npz"))
    world = (np.c_[V, np.ones(len(V))] @ d["tmm"])[:, :3]
    out = os.path.join(S2.OUT, f"{case}_SANITY_stage2.stl")
    trimesh.Trimesh(world, faces, process=False).export(out)
    return out, last

def _cached(c):
    return os.path.exists(os.path.join(S2.CACHE, f"{c}.npz"))

# pick up to 5 TRAINING cases, excluding 12662, that S2 cached — prefer larger
# (better-placed) crowns by design-cloud size so the figure looks clean
cand = [c for c in _tr if _cached(c) and c != "12662"]
def _csize(c):
    return len(np.load(os.path.join(S2.CACHE, f"{c}.npz"))["dpts"])
cand = sorted(cand, key=_csize, reverse=True)[:5]
print("trying training cases:", cand, "\n")

results = []
for pick in cand:
    out, ch = sanity_overfit(pick, steps=250)
    results.append((pick, ch, out))
    print(f"  {pick:26s} final chamfer+lap {ch:.4f}  ->  {os.path.basename(out)}")

results.sort(key=lambda r: r[1])
print("\ntightest fit:", results[0][0], f"(chamfer {results[0][1]:.4f})")
print("open the STLs in out_stage2/ and pick the best-looking one for your figure.")

trying training cases: ['12856', '12885', '12905', '12915-', '12929'] 



/tmp/ipykernel_58/2245659336.py:45: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.from_numpy(t["edges"]).to(DEV), t["faces"])


  12856                      final chamfer+lap 1.2381  ->  12856_SANITY_stage2.stl
  12885                      final chamfer+lap 0.4687  ->  12885_SANITY_stage2.stl
  12905                      final chamfer+lap 0.6020  ->  12905_SANITY_stage2.stl
  12915-                     final chamfer+lap 0.3334  ->  12915-_SANITY_stage2.stl
  12929                      final chamfer+lap 0.4794  ->  12929_SANITY_stage2.stl

tightest fit: 12915- (chamfer 0.3334)
open the STLs in out_stage2/ and pick the best-looking one for your figure.


In [26]:
# %% SAVE EVERYTHING before stopping the kernel --------------------------------
import os, shutil, glob
SAVE = "/kaggle/working/THESIS_SAVE"
if os.path.exists(SAVE): shutil.rmtree(SAVE)
os.makedirs(SAVE)

# 1. the trained models (so you never retrain)
shutil.copytree(CFG.CKPT,  f"{SAVE}/ckpt",  dirs_exist_ok=True)   # Stage 1 best.pt
if os.path.exists("/kaggle/working/ckpt2"):
    shutil.copytree("/kaggle/working/ckpt2", f"{SAVE}/ckpt2", dirs_exist_ok=True)  # Stage 2

# 2. the voxel cache (so you never re-preprocess)
shutil.copytree(CFG.CACHE, f"{SAVE}/cache", dirs_exist_ok=True)

# 3. all results: metrics, STLs, heatmaps, demo, milling shells
for d in ["out", "out_mill", "out_stage2", "demo"]:
    p = f"/kaggle/working/{d}"
    if os.path.exists(p): shutil.copytree(p, f"{SAVE}/{d}", dirs_exist_ok=True)

# zip it for a single clean download
shutil.make_archive("/kaggle/working/THESIS_SAVE", "zip", SAVE)
print("saved -> /kaggle/working/THESIS_SAVE.zip")
print("contents:", os.listdir(SAVE))

saved -> /kaggle/working/THESIS_SAVE.zip
contents: ['ckpt2', 'ckpt', 'cache', 'out_stage2', 'out_mill', 'demo', 'out']


In [45]:
import numpy as np, trimesh
from scipy.spatial import cKDTree

# exocad/3Shape-style occlusal contact scale (heavy -> clearance), thresholds in mm
_OCC_STOPS = [
    (-0.10, (0.55,0.00,0.00)),  # deep penetration   - dark red
    ( 0.00, (0.90,0.10,0.10)),  # contact / heavy     - red
    ( 0.06, (0.98,0.65,0.10)),  # firm-light contact  - orange
    ( 0.10, (0.95,0.92,0.20)),  # light contact       - yellow
    ( 0.15, (0.30,0.80,0.30)),  # ideal proximity     - green
    ( 0.30, (0.20,0.70,0.95)),  # slight clearance    - cyan
    ( 0.60, (0.15,0.25,0.80)),  # clearance / no touch- blue
]
def _occ_color(d):
    xs=[s[0] for s in _OCC_STOPS]; cols=[s[1] for s in _OCC_STOPS]
    d=np.clip(d,xs[0],xs[-1]); rgb=np.zeros((len(d),3))
    for k in range(3): rgb[:,k]=np.interp(d,xs,[c[k] for c in cols])
    return (rgb*255).astype(np.uint8)

def _true_penetration(crown_world, anta_world, V):
    try:
        inter = crown_world.intersection(anta_world)
        if inter is not None and not inter.is_empty and len(inter.vertices) > 0:
            pen_tree = cKDTree(np.asarray(inter.vertices))
            inside = anta_world.contains(V)
            depth = np.zeros(len(V))
            depth[inside] = -pen_tree.query(V[inside])[0] - 1e-4
            return depth, inside
    except Exception as e:
        print("  (intersection backend unavailable, using geometric sign):", repr(e)[:60])
    return None, None

def occlusal_contact_map(crown_world, anta_world, c0, out_ply, tmm=None, legend_png=None):
    """exocad/3Shape occlusal contact map: crown occlusal surface coloured by signed
    distance to the antagonist. Red = heavy contact/penetration ... blue = clearance."""
    apts,_ = trimesh.sample.sample_surface(anta_world, 80000)
    apts=np.asarray(apts); apts=apts[np.linalg.norm(apts-c0,axis=1)<CFG.ROI_MM]
    if len(apts)<20:
        print("  no antagonist points in ROI"); return
    tree=cKDTree(apts); V=np.asarray(crown_world.vertices)
    axis=(np.array([0,0,1,0])@tmm)[:3]; axis/=np.linalg.norm(axis)
    proj=(V-c0)@axis; occ_mask = proj > np.percentile(proj,55)
    d,idx=tree.query(V)

    depth, inside = _true_penetration(crown_world, anta_world, V)
    if depth is not None:
        signed = np.where(inside, depth, d)
    else:
        nn=apts[idx]; pen=((nn-V)@axis)>0
        signed=np.where(pen & (d<0.12), -d, d)

    rgb=np.tile(np.array([[205,205,205]],np.uint8),(len(V),1))
    rgb[occ_mask]=_occ_color(signed[occ_mask])
    m=crown_world.copy(); m.visual.vertex_colors=np.c_[rgb,np.full(len(rgb),255,np.uint8)]
    m.export(out_ply)

    occ=signed[occ_mask]
    heavy=(occ<=0).mean()*100; firm=((occ>0)&(occ<=0.06)).mean()*100
    ideal=((occ>0.10)&(occ<=0.15)).mean()*100
    print(f"  occlusal map exported: {heavy:.0f}% heavy/penetration (red), "
          f"{firm:.0f}% firm contact (orange), {ideal:.0f}% ideal proximity (green)")

    if legend_png:
        import matplotlib; matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.colors import LinearSegmentedColormap, Normalize
        from matplotlib.colorbar import ColorbarBase
        xs=[s[0] for s in _OCC_STOPS]
        norm_stops=[(x-xs[0])/(xs[-1]-xs[0]) for x in xs]
        cmap=LinearSegmentedColormap.from_list("occ",list(zip(norm_stops,[s[1] for s in _OCC_STOPS])))
        fig,ax=plt.subplots(figsize=(1.5,4.2))
        cb=ColorbarBase(ax,cmap=cmap,norm=Normalize(vmin=xs[0]*1000,vmax=xs[-1]*1000),orientation="vertical")
        cb.set_label("crown\u2013antagonist gap (\u00b5m)\nred = heavy contact / penetration",fontsize=9)
        cb.set_ticks([-100,0,60,100,150,300,600])
        fig.tight_layout(); fig.savefig(legend_png,dpi=200,bbox_inches="tight"); plt.close()
        print("  scale bar:",legend_png)

In [46]:
print("occlusal_contact_map" in dir())   # should print True

True


In [47]:
import os, glob
print("FIN.HEAT =", FIN.HEAT)
print("existing plys:", glob.glob(f"{FIN.HEAT}/*.ply")[:5])
print("single-crown attached:", glob.glob("/kaggle/input/**/constructionInfo", recursive=True)[:1])

FIN.HEAT = /kaggle/working/out_mill/fit_heatmap
existing plys: ['/kaggle/working/out_mill/fit_heatmap/2025-10-19_98771-005_fit.ply', '/kaggle/working/out_mill/fit_heatmap/2025-09-30_98773-005_fit.ply', '/kaggle/working/out_mill/fit_heatmap/2025-10-19_98771-003_fit.ply', '/kaggle/working/out_mill/fit_heatmap/2025-09-21_98772-002_fit.ply', '/kaggle/working/out_mill/fit_heatmap/21025_fit.ply']
single-crown attached: ['/kaggle/input/datasets/lamiaaelfadaly/single-crown/content/gdrive/MyDrive/singl crown-copy/16637/constructionInfo']


In [48]:
# %% GENERATE occlusal contact maps for a few test cases -----------------------
import os, numpy as np, torch
os.makedirs(FIN.HEAT, exist_ok=True)

net = UNet3D(in_ch=CFG.IN_CH).to(DEV)
net.load_state_dict(torch.load(f"{CFG.CKPT}/best.pt", map_location=DEV)); net.eval()
_tr,_va,_te = split_by_group(manifest)

pick = _te[:6]                      # first six test cases; adjust as you like
print("generating occlusal maps for:", pick)
for case in pick:
    try:
        pred,gt,tmm = predict_mesh(net, case, DEV)
        ai = crown_world_mesh(pred, tmm)
        design, anta, margin, fdi = _case_world_refs(case)
        c0 = np.asarray(ai.vertices).mean(0)
        occlusal_contact_map(ai, anta, c0,
                             f"{FIN.HEAT}/{case}_occlusion.ply",
                             tmm=tmm,
                             legend_png=f"{FIN.HEAT}/occlusion_scale.png")
    except Exception as e:
        print("  skip", case, repr(e)[:70])
print("done; now run the render cell")

generating occlusal maps for: ['12863', '13013', '13013 - Copy', '13057', '13057 - Copy', '18264']
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 2% heavy/penetration (red), 1% firm contact (orange), 5% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 0% heavy/penetration (red), 0% firm contact (orange), 0% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 0% heavy/penetration (red), 0% firm contact (orange), 0% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueEr

In [49]:
case = "13013"
pred,gt,tmm = predict_mesh(net, case, DEV)
ai = crown_world_mesh(pred, tmm)
design, anta, margin, fdi = _case_world_refs(case)
c0 = np.asarray(ai.vertices).mean(0)

apts,_ = trimesh.sample.sample_surface(anta, 80000); apts=np.asarray(apts)
in_roi = np.linalg.norm(apts-c0,axis=1) < CFG.ROI_MM
print("antagonist pts in ROI:", in_roi.sum(), "of", len(apts))
d = cKDTree(apts[in_roi]).query(np.asarray(ai.vertices))[0] if in_roi.sum() else None
print("gap range (mm):", (round(float(d.min()),2), round(float(d.max()),2)) if d is not None else "no antagonist in ROI")

antagonist pts in ROI: 24671 of 80000
gap range (mm): (0.88, 8.17)


In [50]:
import numpy as np
_tr,_va,_te = split_by_group(manifest)
for case in _te:
    try:
        pred,gt,tmm = predict_mesh(net, case, DEV)
        ai = crown_world_mesh(pred, tmm)
        design, anta, margin, fdi = _case_world_refs(case)
        c0 = np.asarray(ai.vertices).mean(0)
        apts,_ = trimesh.sample.sample_surface(anta, 60000); apts=np.asarray(apts)
        apts = apts[np.linalg.norm(apts-c0,axis=1) < CFG.ROI_MM]
        if len(apts) < 20: 
            print(f"{case:18s} no antagonist"); continue
        gmin = cKDTree(apts).query(np.asarray(ai.vertices))[0].min()
        print(f"{case:18s} min gap {gmin:.2f} mm  {'<-- occludes' if gmin < 0.2 else ''}")
    except Exception as e:
        print(case, repr(e)[:50])

12863              min gap 0.02 mm  <-- occludes
13013              min gap 0.85 mm  
13013 - Copy       min gap 1.99 mm  
13057              min gap 0.01 mm  <-- occludes
13057 - Copy       min gap 0.01 mm  <-- occludes
18264              min gap 0.01 mm  <-- occludes
19099              min gap 0.01 mm  <-- occludes
20094              min gap 0.19 mm  <-- occludes
2024-03-25_00333-001 no antagonist
2025-09-21_98772-002 min gap 0.01 mm  <-- occludes
2025-09-21_98776-001 min gap 0.03 mm  <-- occludes
2025-09-28_98771-003 min gap 0.17 mm  <-- occludes
2025-09-30_98773-004 min gap 6.80 mm  
2025-09-30_98773-005 min gap 0.02 mm  <-- occludes
2025-10-01_98774-002 min gap 0.01 mm  <-- occludes
2025-10-19_98771-003 min gap 0.02 mm  <-- occludes
2025-10-19_98771-005 min gap 0.01 mm  <-- occludes
2025-10-19_98772-004 min gap 1.10 mm  
2025-11-03_98795-006 min gap 0.02 mm  <-- occludes
20377              min gap 0.57 mm  
21025              min gap 0.03 mm  <-- occludes
21062              min ga

In [51]:
good = ['2025-10-19_98771-005','21237','20094','19099','21025',
        '13057','21209','21226','21354','2025-10-01_98774-002']
print("generating occlusal maps for:", good)
for case in good:
    try:
        pred,gt,tmm = predict_mesh(net, case, DEV)
        ai = crown_world_mesh(pred, tmm)
        design, anta, margin, fdi = _case_world_refs(case)
        c0 = np.asarray(ai.vertices).mean(0)
        occlusal_contact_map(ai, anta, c0,
                             f"{FIN.HEAT}/{case}_occlusion.ply",
                             tmm=tmm, legend_png=f"{FIN.HEAT}/occlusion_scale.png")
    except Exception as e:
        print("  skip", case, repr(e)[:70])

generating occlusal maps for: ['2025-10-19_98771-005', '21237', '20094', '19099', '21025', '13057', '21209', '21226', '21354', '2025-10-01_98774-002']
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 2% heavy/penetration (red), 0% firm contact (orange), 3% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 0% heavy/penetration (red), 0% firm contact (orange), 0% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 0% heavy/penetration (red), 0% firm contact (orange), 0% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection

In [52]:
for case in good:
    try:
        pred,gt,tmm = predict_mesh(net, case, DEV)
        ai = crown_world_mesh(pred, tmm)
        ai_smooth = trimesh.smoothing.filter_taubin(ai.copy(), lamb=0.5, nu=-0.53, iterations=30)
        design, anta, margin, fdi = _case_world_refs(case)
        c0 = np.asarray(ai_smooth.vertices).mean(0)
        occlusal_contact_map(ai_smooth, anta, c0,          # <-- smoothed mesh
                             f"{FIN.HEAT}/{case}_occlusion.ply",
                             tmm=tmm, legend_png=f"{FIN.HEAT}/occlusion_scale.png")
    except Exception as e:
        print("  skip", case, repr(e)[:70])

  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 2% heavy/penetration (red), 1% firm contact (orange), 3% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 0% heavy/penetration (red), 0% firm contact (orange), 0% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 0% heavy/penetration (red), 0% firm contact (orange), 0% ideal proximity (green)
  scale bar: /kaggle/working/out_mill/fit_heatmap/occlusion_scale.png
  (intersection backend unavailable, using geometric sign): ValueError('Not all meshes are volumes!')
  occlusal map exported: 0% heavy/penetration (red), 0% firm con

In [35]:
# %% EXTRA EVALUATION — distributions, Bland–Altman, accuracy-vs-size ----------
import os, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

CSV = os.path.join(SAVED, "out", "metrics.csv")   # saved metrics from last run
df  = pd.read_csv(CSV)
OUT = "/kaggle/working/eval_figs"; os.makedirs(OUT, exist_ok=True)
print("rows:", len(df))
print("columns:", list(df.columns))               # <-- check these names match below

rows: 29
columns: ['case', 'fdi', 'IoU', 'Hausdorff_mm', 'RMS_mm', 'occlusal_gap_AI_mm', 'occlusal_gap_design_mm', 'margin_mean_mm', 'margin_max_mm', 'margin_fit_max_mm', 'dev_within_tol_pct']


In [36]:
# helper: find a column by candidate names
def col(*cands):
    for c in cands:
        if c in df.columns: return c
    return None

c_iou  = col("iou","IoU","vox_iou")
c_rms  = col("rms","RMS","rms_mm","rms_surface")
c_hd   = col("hd","hausdorff","HD","hd_mm")
c_marg = col("margin_mean","marg_mean","margin","margin_mm")
c_vol  = col("volume","vol_mm3","crown_volume","vol")
c_fdi  = col("fdi","FDI","tooth")
# occlusal: prefer paired AI/exo gaps for Bland–Altman; else fall back to the |diff|
c_oai  = col("occ_ai","occlusal_ai","occ_gap_ai","gap_ai")
c_oexo = col("occ_exo","occlusal_exo","occ_gap_exo","gap_exo")
c_odiff= col("occ_accuracy","occlusal_accuracy","occ_absdiff","occ_diff")
print("detected:", dict(iou=c_iou,rms=c_rms,hd=c_hd,marg=c_marg,vol=c_vol,
                         fdi=c_fdi,occ_ai=c_oai,occ_exo=c_oexo,occ_diff=c_odiff))

detected: {'iou': 'IoU', 'rms': None, 'hd': None, 'marg': None, 'vol': None, 'fdi': 'fdi', 'occ_ai': None, 'occ_exo': None, 'occ_diff': None}


In [37]:
metrics = [(c_iou,"IoU",False),(c_rms,"RMS (mm)",True),
           (c_hd,"Hausdorff (mm)",True),(c_marg,"Marginal discrepancy (mm)",True)]
metrics = [m for m in metrics if m[0]]
fig,axs = plt.subplots(1,len(metrics),figsize=(3.2*len(metrics),4))
if len(metrics)==1: axs=[axs]
for ax,(c,lab,lower) in zip(axs,metrics):
    v = df[c].dropna().values
    ax.boxplot(v,widths=.5,showfliers=False)
    ax.scatter(np.random.normal(1,0.05,len(v)),v,s=22,alpha=.6,color="#2e6da4",zorder=3)
    ax.set_title(lab,fontsize=10); ax.set_xticks([])
    ax.text(0.5,-0.08,f"mean {v.mean():.3f}\nSD {v.std():.3f}",transform=ax.transAxes,
            ha="center",va="top",fontsize=8)
    if c==c_marg: ax.axhline(0.12,ls="--",color="#b3402f"); 
fig.suptitle("Per-case accuracy across the test set (n=%d)"%len(df),y=1.02,fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/dist_boxplots.png",dpi=200,bbox_inches="tight"); plt.close()
print("saved dist_boxplots.png")

saved dist_boxplots.png


In [38]:
if c_oai and c_oexo:
    ai=df[c_oai].astype(float); exo=df[c_oexo].astype(float)
    mean=(ai+exo)/2; diff=ai-exo
    md=diff.mean(); sd=diff.std()
    lo,hi=md-1.96*sd, md+1.96*sd
    fig,ax=plt.subplots(figsize=(6.5,5))
    ax.scatter(mean,diff,s=30,color="#2e6da4",alpha=.7)
    ax.axhline(md,color="#333"); ax.axhline(lo,ls="--",color="#b3402f"); ax.axhline(hi,ls="--",color="#b3402f")
    for y,t in [(md,f"bias {md:+.3f}"),(hi,f"+1.96 SD {hi:+.3f}"),(lo,f"-1.96 SD {lo:+.3f}")]:
        ax.text(ax.get_xlim()[1],y,t,va="bottom",ha="right",fontsize=9,color="#333")
    ax.set_xlabel("Mean occlusal gap, AI & exocad (mm)")
    ax.set_ylabel("Difference, AI − exocad (mm)")
    ax.set_title("Bland–Altman: occlusal gap agreement")
    fig.tight_layout(); fig.savefig(f"{OUT}/bland_altman_occlusal.png",dpi=200,bbox_inches="tight"); plt.close()
    print(f"saved bland_altman_occlusal.png  (bias {md:+.3f} mm, LoA {lo:+.3f}..{hi:+.3f})")
else:
    print("Bland–Altman skipped: need paired AI & exocad occlusal-gap columns.")
    print("If your CSV only has the |AI−CAD| difference, tell me and I'll plot that as a histogram instead.")

Bland–Altman skipped: need paired AI & exocad occlusal-gap columns.
If your CSV only has the |AI−CAD| difference, tell me and I'll plot that as a histogram instead.


In [39]:
ycol = c_rms or c_iou
if c_vol and ycol:
    v=df[c_vol].astype(float); y=df[ycol].astype(float)
    fig,ax=plt.subplots(figsize=(6.5,5))
    sc=ax.scatter(v,y,s=40,c=y,cmap="viridis",edgecolor="#222")
    # simple trend line
    m,b=np.polyfit(v,y,1); xs=np.linspace(v.min(),v.max(),50)
    ax.plot(xs,m*xs+b,color="#b3402f",lw=1.5,label=f"trend (slope {m:.2e})")
    ax.set_xlabel("Crown volume (mm³)"); ax.set_ylabel(ycol.upper())
    ax.set_title("Accuracy vs crown size"); ax.legend(frameon=False,fontsize=9)
    fig.tight_layout(); fig.savefig(f"{OUT}/accuracy_vs_size.png",dpi=200,bbox_inches="tight"); plt.close()
    r=np.corrcoef(v,y)[0,1]
    print(f"saved accuracy_vs_size.png  (Pearson r = {r:.2f})")
else:
    print("size plot skipped: need a volume column. If volume isn't in the CSV, I can")
    print("compute it per case from the cached crown meshes instead — say the word.")

size plot skipped: need a volume column. If volume isn't in the CSV, I can
compute it per case from the cached crown meshes instead — say the word.


In [27]:
import os, glob

print("Checking important folders/files...\n")

paths = [
    "/kaggle/working/out",
    "/kaggle/working/out/stl",
    "/kaggle/working/out/heatmap",
    "/kaggle/working/out/stl_smooth",
    "/kaggle/working/ckpt",
    "/kaggle/working/cache",
]

for p in paths:
    print(p, "exists:", os.path.exists(p))

print("\nZIP files already present:")
!ls -lh /kaggle/working/*.zip 2>/dev/null || true

Checking important folders/files...

/kaggle/working/out exists: True
/kaggle/working/out/stl exists: True
/kaggle/working/out/heatmap exists: True
/kaggle/working/out/stl_smooth exists: False
/kaggle/working/ckpt exists: True
/kaggle/working/cache exists: True

ZIP files already present:
-rw-r--r-- 1 root root 76M Jun 25 07:53 /kaggle/working/THESIS_SAVE.zip


In [28]:
import os, shutil, zipfile, glob

backup_dir = "/kaggle/working/FINAL_BACKUP_PACKAGE"
zip_path = "/kaggle/working/FINAL_AI_CROWN_COMPLETE_BACKUP.zip"

# Remove old backup folder if it exists
if os.path.exists(backup_dir):
    shutil.rmtree(backup_dir)

os.makedirs(backup_dir, exist_ok=True)

# Copy main folders if they exist
items_to_copy = [
    ("/kaggle/working/out", os.path.join(backup_dir, "out")),
    ("/kaggle/working/ckpt", os.path.join(backup_dir, "ckpt")),
]

for src, dst in items_to_copy:
    if os.path.exists(src):
        shutil.copytree(src, dst)
        print("Copied:", src)
    else:
        print("Missing, skipped:", src)

# Copy manifest only from cache
cache_backup = os.path.join(backup_dir, "cache")
os.makedirs(cache_backup, exist_ok=True)

manifest_src = "/kaggle/working/cache/manifest.json"
if os.path.exists(manifest_src):
    shutil.copy(manifest_src, os.path.join(cache_backup, "manifest.json"))
    print("Copied manifest.json")
else:
    print("manifest.json not found, skipped")

# Copy useful CSVs
csv_files = [
    "/kaggle/working/final_experiment_comparison.csv",
    "/kaggle/working/out/metrics.csv",
]

for src in csv_files:
    if os.path.exists(src):
        shutil.copy(src, backup_dir)
        print("Copied:", src)
    else:
        print("Missing, skipped:", src)

# Create README safely with normal ASCII only
readme_text = """
FINAL AI CROWN MODEL BACKUP

Final selected model:
- RES = 128
- Input channels = 3
  1. preparation shell
  2. antagonist shell
  3. margin channel
- Threshold = 0.5
- Checkpoint = ckpt/best.pt

Final selected results:
- IoU approximately 0.6200
- Hausdorff distance approximately 1.73 mm
- RMS surface deviation approximately 0.486 mm
- Margin mean discrepancy approximately 0.089 mm
- Margin max discrepancy approximately 0.178 mm
- Occlusal accuracy |AI-exocad gap| approximately 0.1285 mm

Folders:
- out/stl = raw AI crown STL files
- out/heatmap = PLY deviation heatmaps
- out/stl_smooth = smoothed visualization STL crowns, if generated
- ckpt/best.pt = trained final checkpoint
- cache/manifest.json = case manifest
"""

with open(os.path.join(backup_dir, "README_FINAL_MODEL.txt"), "w") as f:
    f.write(readme_text)

print("README created.")

# Create zip
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(backup_dir):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, "/kaggle/working")
            zf.write(full_path, rel_path)

print("\nCreated full backup zip:")
print(zip_path)

!ls -lh /kaggle/working/FINAL_AI_CROWN_COMPLETE_BACKUP.zip

Copied: /kaggle/working/out
Copied: /kaggle/working/ckpt
Copied manifest.json
Missing, skipped: /kaggle/working/final_experiment_comparison.csv
Copied: /kaggle/working/out/metrics.csv
README created.

Created full backup zip:
/kaggle/working/FINAL_AI_CROWN_COMPLETE_BACKUP.zip
-rw-r--r-- 1 root root 43M Jun 25 07:53 /kaggle/working/FINAL_AI_CROWN_COMPLETE_BACKUP.zip


In [29]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/FINAL_AI_CROWN_COMPLETE_BACKUP.zip"))

/kaggle/working/FINAL_AI_CROWN_COMPLETE_BACKUP.zip

In [30]:
from collections import Counter
import glob, os
types = []
for ci in glob.glob(f"{CFG.INPUT_ROOT}/**/constructionInfo", recursive=True) or \
         glob.glob("/kaggle/input/**/constructionInfo", recursive=True):
    for t in parse_construction_info(ci):
        rt = None
        try:
            import xml.etree.ElementTree as ET
            for tooth in ET.parse(ci).getroot().findall("./Teeth/Tooth"):
                e = tooth.find("ReconstructionType")
                if e is not None: types.append(e.text)
        except Exception:
            pass
print(Counter(types))

Counter({'AnatomicCrown': 437, 'Antagonist': 12})


In [31]:
import glob, os
cis = glob.glob(f"{CFG.INPUT_ROOT}/**/constructionInfo", recursive=True) or \
      glob.glob("/kaggle/input/**/constructionInfo", recursive=True)
print("case folders:", len(cis))

multi = 0
total_teeth = 0
for ci in cis:
    folder = os.path.dirname(ci)
    teeth = parse_construction_info(ci)
    total_teeth += len(teeth)
    designs = glob.glob(os.path.join(folder, "*design*.stl")) + \
              glob.glob(os.path.join(folder, "*crown*.stl"))
    if len(teeth) > 1:
        multi += 1
        if multi <= 5:  # show a few examples
            print(f"\n{os.path.basename(folder)}")
            print(f"   teeth in constructionInfo: {[t['fdi'] for t in teeth]}")
            print(f"   design/crown .stl files:   {[os.path.basename(d) for d in designs]}")
            print(f"   all .stl in folder:        {[os.path.basename(f) for f in glob.glob(os.path.join(folder,'*.stl'))]}")

print(f"\nfolders: {len(cis)} | total teeth listed: {total_teeth} | folders with >1 tooth: {multi}")

case folders: 290

2024-07-15_00224-004 - Copy
   teeth in constructionInfo: [46, 36]
   design/crown .stl files:   ['design.stl']
   all .stl in folder:        ['design.stl']

2024-01-22_00071-005 - Copy
   teeth in constructionInfo: [11, 21]
   design/crown .stl files:   ['design.stl']
   all .stl in folder:        ['design.stl']

2024-09-22_00296-002 - Copy
   teeth in constructionInfo: [24, 25]
   design/crown .stl files:   ['design.stl']
   all .stl in folder:        ['design.stl']

2024-07-16_00228-002
   teeth in constructionInfo: [25, 46]
   design/crown .stl files:   ['design 2.stl', 'design.stl']
   all .stl in folder:        ['design 2.stl', 'design.stl']

14633 - Copy
   teeth in constructionInfo: [11, 21]
   design/crown .stl files:   ['design.stl']
   all .stl in folder:        ['design.stl']

folders: 290 | total teeth listed: 339 | folders with >1 tooth: 43


In [32]:
import glob, os, numpy as np
cis = glob.glob(f"{CFG.INPUT_ROOT}/**/constructionInfo", recursive=True) or \
      glob.glob("/kaggle/input/**/constructionInfo", recursive=True)

stats = {"folders":0,"no_design":0,"no_tooth":0,"no_tmm":0,"no_arch":0,"ok":0,"extra_designs":0}
for ci in cis:
    stats["folders"] += 1
    folder = os.path.dirname(ci)
    designs = sorted(glob.glob(os.path.join(folder,"*.stl")))
    designs = [d for d in designs if "design" in os.path.basename(d).lower()]
    if not designs: stats["no_design"] += 1; continue
    teeth = parse_construction_info(ci)
    if not teeth: stats["no_tooth"] += 1; continue
    if len(designs) > 1: stats["extra_designs"] += len(designs) - 1
    # arch file check for the matched tooth
    from numpy.linalg import norm
    d0 = _load(designs[0]); dcen = d0.vertices.mean(0)
    tooth = min(teeth, key=lambda t: np.inf if t["center"] is None else norm(t["center"]-dcen))
    if tooth["tmm"] is None: stats["no_tmm"] += 1; continue
    arch = fdi_arch(tooth["fdi"])
    prep = os.path.join(folder, f"{arch}.stl")
    anta = os.path.join(folder, f"{'lower' if arch=='upper' else 'upper'}.stl")
    if not (os.path.exists(prep) and os.path.exists(anta)):
        stats["no_arch"] += 1
        print("  missing arch in:", os.path.basename(folder),
              "| has:", [os.path.basename(f) for f in glob.glob(os.path.join(folder,'*.stl'))])
        continue
    stats["ok"] += 1
print(stats)

  missing arch in: 16637 | has: ['design.stl']
  missing arch in: 2025-08-13_00758-002 | has: ['design.stl']
  missing arch in: 17504 | has: ['design.stl']
  missing arch in: 2024-07-15_00224-004 - Copy | has: ['design.stl']
  missing arch in: 16881 | has: ['design.stl']
  missing arch in: 15692 | has: ['design.stl']
  missing arch in: 2024-07-21_00241-005 | has: ['design.stl']
  missing arch in: 16515 | has: ['design.stl']
  missing arch in: 2024-01-22_00071-005 - Copy | has: ['design.stl']
  missing arch in: 14381 | has: ['design.stl']
  missing arch in: 17255 | has: ['design.stl']
  missing arch in: 14680 | has: ['design.stl']
  missing arch in: 19197 | has: ['design.stl']
  missing arch in: 16281 | has: ['design.stl']
  missing arch in: 16530 | has: ['design.stl']
  missing arch in: 2024-09-22_00296-002 - Copy | has: ['design.stl']
  missing arch in: 16116 | has: ['design.stl']
  missing arch in: 19309 | has: ['design.stl']
  missing arch in: 16240 | has: ['design.stl']
  missing a

In [33]:
# %% S5 — train Stage 2 --------------------------------------------------------
def train_stage2():
    net = DeformNet().to(DEV)
    opt = torch.optim.Adam(net.parameters(), S2.LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, S2.EPOCHS)
    best = 1e9
    for ep in range(S2.EPOCHS):
        net.train(); random.shuffle(_tr)
        for c in _tr:
            g = _grid4(c).to(DEV)
            tv, ed, _ = _tmpl(c)
            G = torch.from_numpy(np.load(os.path.join(S2.CACHE, f"{c}.npz"))["dpts"]).to(DEV)
            opt.zero_grad()
            P, off = net(g, tv)
            Pi = P[torch.randint(len(P), (S2.N_CHAMF,), device=DEV)]
            Gi = G[torch.randint(len(G), (S2.N_CHAMF,), device=DEV)]
            loss = chamfer(Pi, Gi) + S2.LAMBDA_LAP * lap_smooth(off, ed)
            loss.backward(); opt.step()
        sched.step()
        net.eval(); vs = []
        with torch.no_grad():
            for c in _va:
                g = _grid4(c).to(DEV); tv, ed, _ = _tmpl(c)
                G = torch.from_numpy(np.load(os.path.join(S2.CACHE, f"{c}.npz"))["dpts"]).to(DEV)
                P, _ = net(g, tv)
                vs.append(chamfer(P[:S2.N_CHAMF], G[:S2.N_CHAMF]).item())
        v = float(np.mean(vs)) if vs else 0.0
        if ep % 10 == 0 or ep == S2.EPOCHS - 1:
            print(f"epoch {ep:3d}  val chamfer {v:.4f}")
        if v < best:
            best = v; torch.save(net.state_dict(), f"{S2.CKPT}/deform_best.pt")
    print(f"best val chamfer {best:.4f} -> {S2.CKPT}/deform_best.pt")

train_stage2()


epoch   0  val chamfer 3.2498


KeyboardInterrupt: 

In [ ]:
# %% S6 — evaluate Stage 2 (reuses your metric functions) ----------------------
@torch.no_grad()
def stage2_world_mesh(net, case):
    g = _grid4(case).to(DEV); tv, _, faces = _tmpl(case)
    V = net(g, tv)[0].cpu().numpy()
    tmm = np.load(os.path.join(CFG.CACHE, f"{case}.npz"))["tmm"]
    world = (np.c_[V, np.ones(len(V))] @ tmm)[:, :3]
    return trimesh.Trimesh(world, faces, process=False), tmm

def evaluate_stage2(test_cases):
    net = DeformNet().to(DEV)
    net.load_state_dict(torch.load(f"{S2.CKPT}/deform_best.pt", map_location=DEV)); net.eval()
    stl = os.path.join(S2.OUT, "stl"); heat = os.path.join(S2.OUT, "heatmap")
    os.makedirs(stl, exist_ok=True); os.makedirs(heat, exist_ok=True)
    rows = []
    for c in test_cases:
        ai, tmm = stage2_world_mesh(net, c)
        design, anta, margin, fdi = _case_world_refs(c)
        ai.export(os.path.join(stl, f"{c}_AI_crown_stage2.stl"))
        fitted = snap_to_margin(ai, margin, tmm)
        fitted.export(os.path.join(stl, f"{c}_AI_crown_stage2_fitted.stl"))
        hd, rms = surface_metrics(ai, design)
        c0 = np.asarray(ai.vertices).mean(0)
        og_ai = occlusal_gap(ai, anta, c0); og_de = occlusal_gap(design, anta, c0)
        mg_mean, mg_max = margin_discrepancy(ai, margin)
        _, pct_tol, pct02 = deviation_heatmap(ai, design, os.path.join(heat, f"{c}_dev.ply"))
        rows.append({"case": c, "fdi": fdi, "Hausdorff_mm": round(hd, 4), "RMS_mm": round(rms, 4),
                     "occlusal_gap_AI_mm": round(og_ai, 4), "occlusal_gap_design_mm": round(og_de, 4),
                     "margin_mean_mm": round(mg_mean, 4), "margin_max_mm": round(mg_max, 4),
                     "within_0.1_pct": round(pct_tol, 1), "within_0.2_pct": round(pct02, 1)})
        print(f"  {c:26s} RMS {rms:.2f} HD {hd:.2f} occl(AI/exo {og_ai:.2f}/{og_de:.2f}) "
              f"margin {mg_mean:.2f}/{mg_max:.2f} <0.2mm {pct02:.0f}%")
    import csv
    keys = list(rows[0].keys())
    with open(f"{S2.OUT}/metrics_stage2.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
    print("\n=== STAGE 2 test summary (vs Stage 1) ===")
    for k in keys[2:]:
        col = np.array([r[k] for r in rows], float); col = col[~np.isnan(col)]
        if len(col):
            print(f"  {k:20s} mean {col.mean():.4f}  SD {col.std(ddof=1):.4f}")
    print(f"  anatomical STLs: {stl}/   (*_stage2.stl morphology, *_fitted.stl seatable)")
    return rows

metrics_stage2 = evaluate_stage2(_te)


In [53]:
# ============================================================
# FINAL DOWNLOAD PACKAGE — ALL AI CROWN OUTPUTS
# ============================================================

import os
import zipfile
import pandas as pd
from pathlib import Path
from IPython.display import FileLink, display

ROOT = Path("/kaggle/working")
ZIP_PATH = ROOT / "AI_CROWN_ALL_OUTPUTS_FINAL_DOWNLOAD.zip"
MANIFEST_PATH = ROOT / "AI_CROWN_ALL_OUTPUTS_MANIFEST.csv"

# Include these folders because they contain your real outputs
INCLUDE_FOLDERS = [
    "FINAL_BACKUP_PACKAGE",
    "THESIS_SAVE",
    "cache",
    "ckpt",
    "ckpt2",
    "demo",
    "eval_figs",
    "out",
    "out_mill",
    "out_stage2",
    "stage2cache",
]

# Include important files directly in /kaggle/working
INCLUDE_FILE_EXTENSIONS = [
    ".csv", ".txt", ".json", ".pt", ".pth",
    ".stl", ".ply", ".png", ".jpg", ".jpeg",
    ".zip", ".ipynb"
]

# Avoid including the new final zip inside itself
SKIP_NAMES = {
    ZIP_PATH.name,
}

files_to_zip = []

for item in ROOT.iterdir():
    if item.name in SKIP_NAMES:
        continue

    if item.is_dir() and item.name in INCLUDE_FOLDERS:
        for p in item.rglob("*"):
            if p.is_file():
                files_to_zip.append(p)

    elif item.is_file():
        if item.suffix.lower() in INCLUDE_FILE_EXTENSIONS:
            files_to_zip.append(item)

# Remove duplicates
files_to_zip = sorted(set(files_to_zip))

manifest_rows = []
for p in files_to_zip:
    rel = p.relative_to(ROOT).as_posix()
    size_mb = p.stat().st_size / (1024 * 1024)
    manifest_rows.append({
        "relative_path": rel,
        "size_MB": round(size_mb, 3),
        "extension": p.suffix.lower()
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False)

print("Files included:", len(files_to_zip))
print("Total size before compression:", round(manifest["size_MB"].sum(), 2), "MB")
display(manifest.head(50))

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    zf.write(MANIFEST_PATH, "AI_CROWN_ALL_OUTPUTS_MANIFEST.csv")

    for p in files_to_zip:
        rel = p.relative_to(ROOT).as_posix()
        zf.write(p, rel)

print("\nCreated final download package:")
print(ZIP_PATH)
print("ZIP size:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2), "MB")

display(FileLink(str(ZIP_PATH)))

Files included: 698
Total size before compression: 1390.45 MB


,relative_path,size_MB,extension
0,FINAL_AI_CROWN_COMPLETE_BACKUP.zip,42.896,.zip
1,FINAL_BACKUP_PACKAGE/README_FINAL_MODEL.txt,0.001,.txt
2,FINAL_BACKUP_PACKAGE/cache/manifest.json,0.019,.json
3,FINAL_BACKUP_PACKAGE/ckpt/best.pt,5.362,.pt
4,FINAL_BACKUP_PACKAGE/metrics.csv,0.002,.csv
5,FINAL_BACKUP_PACKAGE/out/heatmap/12863_dev.ply,0.599,.ply
6,FINAL_BACKUP_PACKAGE/out/heatmap/13013 - Copy_...,0.854,.ply
7,FINAL_BACKUP_PACKAGE/out/heatmap/13013_dev.ply,0.523,.ply
8,FINAL_BACKUP_PACKAGE/out/heatmap/13057 - Copy_...,0.710,.ply
9,FINAL_BACKUP_PACKAGE/out/heatmap/13057_dev.ply,0.516,.ply



Created final download package:
/kaggle/working/AI_CROWN_ALL_OUTPUTS_FINAL_DOWNLOAD.zip
ZIP size: 977.52 MB


/kaggle/working/AI_CROWN_ALL_OUTPUTS_FINAL_DOWNLOAD.zip